In [1]:
from pathlib import Path

import imagecodecs
import imagecodecs.zarr as codecs
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import qim3d
import qim3d.io
import seaborn as sns
import zarr

from comptest.lossless_tests import BASE_JPEGXL_CONFIG
from comptest.zarr import ZarrCompressionConfig, compression_test

%load_ext autoreload
%autoreload 2

In [2]:
imagecodecs.zarr.register_codecs()

In [3]:
DANFIX_201 = Path("/dtu/3d-imaging-center/projects/2026_DANFIX_201")

COMPRESSION_TESTS_DIR = DANFIX_201 / "analysis" / "compression" / "compression_tests"
assert COMPRESSION_TESTS_DIR.is_dir()

DATASETS_DIR = DANFIX_201 / "analysis" / "compression" / "datasets"
assert DATASETS_DIR.is_dir()

VOLUME_COMPRESSION_TESTS_DIR = COMPRESSION_TESTS_DIR / "volume_compression_tests"
assert VOLUME_COMPRESSION_TESTS_DIR.is_dir()

In [4]:
DS_NAME = "Canislupus_CZM_712"

# Load as numpy array
ds = qim3d.io.load(DATASETS_DIR / DS_NAME, contains=DS_NAME + "_")
print(ds.shape)


(1572, 2000, 2000)


In [5]:
# # Rearrange away from planar view to (x, y, view) for compression tests
# ds = einops.rearrange(ds, "view x y -> x y view")
# print(ds.shape)

In [6]:
# DS Tiff file size
tif_total_size = 0
for tif_file in (DATASETS_DIR / DS_NAME).glob("*.tif"):
    tif_total_size += tif_file.stat().st_size

print(tif_total_size, "bytes")
print(tif_total_size / 1e9, "GB")

12576198072 bytes
12.576198072 GB


In [7]:
# from comptest.lossless_tests import C


default_serializer = (
    zarr.codecs.BytesCodec,
    {},
)

chunks_1_channel = (1, 1000, 1000)  # 16-bit * 1 * 1000 * 1000 = 2 MB per chunk
chunks_3_channel = (3, 500, 500)  # 16-bit * 3 * 500 * 500 = 1.5 MB per chunk
chunks_4_channel = (4, 500, 500)  # 16-bit * 4 * 500 * 500 = 2 MB per chunk
chunks_131_channel = (131, 100, 100)  # 16-bit * 1 * 131 * 1000 = 2.62 MB per chunk
chunks_big_image_1_channel = (
    1,
    2000,
    2000,
)  # 16-bit * 1 * 2000 * 2000 = 8 MB per chunk
chunks_big_image_5_channel = (
    5,
    2000,
    2000,
)  # 16-bit * 5 * 2000 * 2000 = 40 MB per chunk
chunks_default = (3, 2000, 2000)

CONFIGS = [
    # {
    #     "serializer": C(codecs.Jpegxl, {"level": 5, "effort": 5, "lossless": True}),
    #     "chunks": chunks_big_image_1_channel,
    # },
    # {
    #     "serializer": C(codecs.Png, {}),
    #     "chunks": chunks_big_image_1_channel,
    #     "lossless": True,
    # },
    # ZarrCompressionConfig(
    #     is_lossless=True,
    #     serializer=(
    #         codecs.Png,
    #         {},
    #     ),
    #     chunks=chunks_big_image_1_channel,
    # ),
    # {
    #     "serializer": C(codecs.Jpeg2k, {"level": 0, "reversible": True}),
    #     "chunks": chunks_default,
    # },
    # {
    #     "serializer": C(codecs.Jpegls, {"level": 0}),
    #     "chunks": chunks_default,
    # },
    # {
    #     "serializer": C(codecs.Webp, {"lossless": True, "method": 4}),
    #     "chunks": chunks_default,
    # },
    ZarrCompressionConfig(
        is_lossless=True,
        series="test",
        serializer=(
            codecs.Jpegxl,
            {"effort": 2, "lossless": True} | BASE_JPEGXL_CONFIG,
        ),
        chunks=(1, ..., ...),
    )
]

In [ ]:
# http://localhost:8888/proxy/8787/status

In [8]:
tests = []
for config in CONFIGS:
    print(f"Running compression test for {config}")
    test = compression_test(
        dataset_name=DS_NAME,
        dataset_path=DATASETS_DIR / DS_NAME,
        data=ds,
        config=config,
        baseline=None,
    )

    tests.append(test)

Running compression test for name=None series='test' is_lossless=True chunks=(1, Ellipsis, Ellipsis) filters=None serializer=(<class 'imagecodecs.zarr.Jpegxl'>, {'effort': 2, 'lossless': True, 'level': 5, 'planar': True, 'transfer': <TRANSFER_FUNCTION.LINEAR: 8>, 'photometric': <COLOR_SPACE.GRAY: 1>, 'usecontainer': False}) compressors=None


In [2]:
import zarr
import zarr.storage
from imagecodecs import zarr as codecs

codecs.register_codecs()

dtype = np.uint16
shape = (100, 2000, 2000)

arr = np.random.randint(0, 65536, size=shape, dtype=dtype)

z = zarr.create_array(
    store=zarr.storage.MemoryStore(),
    shape=arr.shape,
    dtype=arr.dtype,
    chunks=(1, 2000, 2000),
    serializer=(codecs.Jpegxl(planar=True, effort=2, lossless=True)),
)

z[:] = arr


ValueError: planar requires at least 3 dimensions